In [1]:
import pandas as pd

pd_2 = pd.read_excel("Copy of PO# VAL81126.xlsx")

pd_2

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11
0,PO ISSUE DATE :-,NaN,NaN,AUGUST 11 2026,NaN,2nd London Closing of 12-08-2026,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,OMJ Style #,Elegant Jewelry Style #,Quantity,Metal Type,Metal Color,Shank Thickness,Shank Width,PO #,Diamod Quality,Size,Finish Type #,Delivery Time
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,VAL1519-30,YR4171E,25,14K,WHITE GOLD,1.37,1.95,VAL81126,HI-W5A,7,complete,10 DAYS
6,VAL1915-50,YR4171F,20,14K,WHITE GOLD,1.36,2.44,VAL81126,HI-W5A,7,complete,10 DAYS
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
import pandas as pd

def find_header_row(filepath, key_column_name):
    """
    Reads the raw Excel sheet without headers and searches for the row
    that contains the specified key column name.
    """
    # Read sheet without assuming any row as header
    raw_df = pd.read_excel(filepath, header=None)
    
    # Scan rows to find which index contains the key column name
    for row_idx, row in raw_df.iterrows():
        # Convert row values to string and strip whitespace
        row_values = row.astype(str).str.strip().values
        if key_column_name in row_values:
            return row_idx
            
    raise ValueError(f"Could not find header '{key_column_name}' in file: {filepath}")


def process_jewelry_files(input_filepath, master_filepath, output_filepath):
    # 1. Dynamically locate header rows
    input_header_idx = find_header_row(input_filepath, 'Elegant Jewelry Style #')
    master_header_idx = find_header_row(master_filepath, 'Style No')

    # 2. Load the Excel files using the detected header rows
    input_df = pd.read_excel(input_filepath, header=input_header_idx)
    master_df = pd.read_excel(master_filepath, header=master_header_idx)

    # 3. Clean column headers (remove leading/trailing spaces)
    input_df.columns = input_df.columns.astype(str).str.strip()
    master_df.columns = master_df.columns.astype(str).str.strip()

    # 4. Define the Metal Color mapping dictionary
    color_map = {
        'white gold': 'w',
        'yellow gold': 'y',
        'pink gold': 'p',
        'platinum': 'pt',
        'alloy': 'al'
    }

    # 5. Prepare matching columns (ignoring case & whitespace)
    input_df['match_color'] = (
        input_df['Metal Color']
        .astype(str)
        .str.strip()
        .str.lower()
        .map(color_map)
    )

    input_df['match_style'] = input_df['Elegant Jewelry Style #'].astype(str).str.strip()
    input_df['match_size'] = input_df['Size'].astype(str).str.strip()

    master_df['match_style'] = master_df['Style No'].astype(str).str.strip()
    master_df['match_color'] = master_df['COLOR'].astype(str).str.strip().str.lower()
    master_df['match_size'] = master_df['SIZE'].astype(str).str.strip()

    # 6. Merge Input and Master DataFrames on the matching keys
    merged_df = pd.merge(
        input_df,
        master_df[['match_style', 'match_color', 'match_size', 'Client Style No']],
        on=['match_style', 'match_color', 'match_size'],
        how='left'
    )

    # 7. Drop temporary matching columns
    output_df = merged_df.drop(columns=['match_style', 'match_color', 'match_size'])

    # 8. Save the output
    output_df.to_excel(output_filepath, index=False)
    print(f"Processing complete! Results saved to '{output_filepath}'.")


# Example Usage
if __name__ == "__main__":
    process_jewelry_files(
        input_filepath=r"D:\latest\omj\Copy of PO# VAL81126.xlsx",
        master_filepath=r"C:\Users\Pratik.SJFS\Downloads\OMJ_CS_1408.xlsx",
        output_filepath='output_with_client_style.xlsx'
    )

Processing complete! Results saved to 'output_with_client_style.xlsx'.


In [13]:
import pandas as pd


def find_header_row(filepath, key_column_name):
    """Reads the raw Excel sheet without headers and searches for the row that

    contains the specified key column name.
    """
    raw_df = pd.read_excel(filepath, header=None)

    for row_idx, row in raw_df.iterrows():
        row_values = row.astype(str).str.strip().values
        if key_column_name in row_values:
            return row_idx

    raise ValueError(
        f"Could not find header '{key_column_name}' in file: {filepath}"
    )


# --- Strict Cleaning Helpers ---
def clean_style(series):
    """Trims whitespace and converts style numbers to uppercase."""
    return series.astype(str).str.strip().str.upper()


def clean_size(series):
    """Trims whitespace and converts float strings like '7.0' to '7'."""
    return (
        series.astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.lower()
    )


def clean_color(series):
    """Standardizes color names and abbreviations across both files."""
    color_map = {
        "white gold": "w",
        "white": "w",
        "wg": "w",
        "w": "w",
        "yellow gold": "y",
        "yellow": "y",
        "yg": "y",
        "y": "y",
        "pink gold": "p",
        "rose gold": "p",
        "pg": "p",
        "rg": "p",
        "p": "p",
        "platinum": "pt",
        "plat": "pt",
        "pt": "pt",
        "alloy": "al",
        "al": "al",
    }
    cleaned = series.astype(str).str.strip().str.lower()
    return cleaned.map(color_map).fillna(cleaned)


def process_jewelry_files(input_filepath, master_filepath):
    # 1. Dynamically locate header rows
    input_header_idx = find_header_row(
        input_filepath, "Elegant Jewelry Style #"
    )
    master_header_idx = find_header_row(master_filepath, "Style No")

    # 2. Load the Excel files using detected header rows
    input_df = pd.read_excel(input_filepath, header=input_header_idx)
    master_df = pd.read_excel(master_filepath, header=master_header_idx)

    # 3. Clean column headers
    input_df.columns = input_df.columns.astype(str).str.strip()
    master_df.columns = master_df.columns.astype(str).str.strip()

    # 4. Strictly normalize Input DataFrame keys
    input_df["match_style"] = clean_style(input_df["Elegant Jewelry Style #"])
    input_df["match_color"] = clean_color(input_df["Metal Color"])
    input_df["match_size"] = clean_size(input_df["Size"])

    # 5. Strictly normalize Master DataFrame keys (using same logic)
    master_df["match_style"] = clean_style(master_df["Style No"])
    master_df["match_color"] = clean_color(master_df["COLOR"])
    master_df["match_size"] = clean_size(master_df["SIZE"])

    # 6. Remove duplicate keys from Master to guarantee a 1-to-1 match
    master_cleaned = master_df[
        ["match_style", "match_color", "match_size", "Client Style No"]
    ].drop_duplicates(
        subset=["match_style", "match_color", "match_size"], keep="first"
    )

    # 7. Strictly merge on all 3 matching keys
    merged_df = pd.merge(
        input_df,
        master_cleaned,
        on=["match_style", "match_color", "match_size"],
        how="left",
    )

    # 8. Output only the Client Style No column
    return merged_df[["Client Style No"]]


# Example Usage
if __name__ == "__main__":
    result_df = process_jewelry_files(
        input_filepath=r"D:\ops\tools\data\Om jewelery\PO# STOCK9925.xlsx",
        master_filepath=r"C:\Users\Pratik.SJFS\Downloads\OMJ_CS_1408.xlsx",
    )

    print(result_df)
    result_df.to_excel('output1.xlsx')

   Client Style No
0      YR4171B-7WG
1      YR4171C-7WG
2      YR4171E-7WG
3      YR4171Z-7WG
4      YR4171F-7WG
..             ...
59             NaN
60             NaN
61             NaN
62             NaN
63             NaN

[64 rows x 1 columns]


In [ ]:
import pandas as pd


def find_header_row(filepath, key_column_name):
    """Reads the raw Excel sheet without headers and searches for the row that

    contains the specified key column name.
    """
    raw_df = pd.read_excel(filepath, header=None)

    for row_idx, row in raw_df.iterrows():
        row_values = row.astype(str).str.strip().values
        if key_column_name in row_values:
            return row_idx

    raise ValueError(
        f"Could not find header '{key_column_name}' in file: {filepath}"
    )


# --- Strict Cleaning Helpers ---
def clean_style(series):
    """Trims whitespace and converts style numbers to uppercase."""
    return series.astype(str).str.strip().str.upper()


def clean_size(series):
    """Trims whitespace and converts float strings like '7.0' to '7'."""
    return (
        series.astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.lower()
    )


def clean_color(series):
    """Standardizes color names and abbreviations across both files."""
    color_map = {
        "white gold": "w",
        "white": "w",
        "wg": "w",
        "w": "w",
        "yellow gold": "y",
        "yellow": "y",
        "yg": "y",
        "y": "y",
        "pink gold": "p",
        "rose gold": "p",
        "pg": "p",
        "rg": "p",
        "p": "p",
        "platinum": "pt",
        "plat": "pt",
        "pt": "pt",
        "alloy": "al",
        "al": "al",
    }
    cleaned = series.astype(str).str.strip().str.lower()
    return cleaned.map(color_map).fillna(cleaned)


def has_sm_suffix(series):
    """Checks if the string ends with 'SM' (case-insensitive)."""
    return series.astype(str).str.strip().str.upper().str.endswith("SM")


def process_jewelry_files(input_filepath, master_filepath):
    # 1. Dynamically locate header rows
    input_header_idx = find_header_row(
        input_filepath, "Elegant Jewelry Style #"
    )
    master_header_idx = find_header_row(master_filepath, "Style No")

    # 2. Load the Excel files using detected header rows
    input_df = pd.read_excel(input_filepath, header=input_header_idx)
    master_df = pd.read_excel(master_filepath, header=master_header_idx)

    # 3. Clean column headers (strip spaces)
    input_df.columns = input_df.columns.astype(str).str.strip()
    master_df.columns = master_df.columns.astype(str).str.strip()

    # 4. Strictly normalize Input DataFrame keys
    input_df["match_style"] = clean_style(input_df["Elegant Jewelry Style #"])
    input_df["match_color"] = clean_color(input_df["Metal Color"])
    input_df["match_size"] = clean_size(input_df["Size"])

    # Extract SM suffix condition from 'OMJ Style #' if present
    if "OMJ Style #" in input_df.columns:
        input_df["match_sm"] = has_sm_suffix(input_df["OMJ Style #"])
    else:
        input_df["match_sm"] = False

    # 5. Strictly normalize Master DataFrame keys
    master_df["match_style"] = clean_style(master_df["Style No"])
    master_df["match_color"] = clean_color(master_df["COLOR"])
    master_df["match_size"] = clean_size(master_df["SIZE"])

    # Extract SM suffix condition from Master's 'Client Style No'
    master_df["match_sm"] = has_sm_suffix(master_df["Client Style No"])

    # 6. Remove duplicate keys from Master (including match_sm)
    merge_keys = ["match_style", "match_color", "match_size", "match_sm"]

    master_cleaned = master_df[
        merge_keys + ["Client Style No"]
    ].drop_duplicates(subset=merge_keys, keep="first")

    # 7. Merge on all 4 keys (Style, Color, Size, SM condition)
    merged_df = pd.merge(input_df, master_cleaned, on=merge_keys, how="left")

    # 8. Output only the Client Style No column
    return merged_df[["Client Style No"]]


# Example Usage
if __name__ == "__main__":
    result_df = process_jewelry_files(
        input_filepath=r"D:\ops\tools\data\Om jewelery\PO# STOCK9925.xlsx",
        master_filepath=r"C:\Users\Pratik.SJFS\Downloads\OMJ_CS_1408.xlsx",
    )

    print(result_df)
    result_df.to_excel('output2.xlsx')




    ## final working logic

   Client Style No
0      YR4171B-7WG
1      YR4171C-7WG
2      YR4171E-7WG
3      YR4171Z-7WG
4      YR4171F-7WG
..             ...
59             NaN
60             NaN
61             NaN
62             NaN
63             NaN

[64 rows x 1 columns]


In [18]:
import pandas as pd


def find_header_row(filepath, key_column_name):
    """Reads the raw Excel sheet without headers and searches for the row that

    contains the specified key column name.
    """
    raw_df = pd.read_excel(filepath, header=None)

    for row_idx, row in raw_df.iterrows():
        row_values = row.astype(str).str.strip().values
        if key_column_name in row_values:
            return row_idx

    raise ValueError(
        f"Could not find header '{key_column_name}' in file: {filepath}"
    )


# --- Strict Cleaning Helpers ---
def clean_style(series):
    """Trims whitespace and converts style numbers to uppercase."""
    return series.astype(str).str.strip().str.upper()


def clean_size(series):
    """Trims whitespace and converts float strings like '7.0' to '7'."""
    return (
        series.astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.lower()
    )


def clean_color(series):
    """Standardizes color names and abbreviations across both files."""
    color_map = {
        "white gold": "w",
        "white": "w",
        "wg": "w",
        "w": "w",
        "yellow gold": "y",
        "yellow": "y",
        "yg": "y",
        "y": "y",
        "pink gold": "p",
        "rose gold": "p",
        "pg": "p",
        "rg": "p",
        "p": "p",
        "platinum": "pt",
        "plat": "pt",
        "pt": "pt",
        "alloy": "al",
        "al": "al",
    }
    cleaned = series.astype(str).str.strip().str.lower()
    return cleaned.map(color_map).fillna(cleaned)


def has_sm_suffix(series):
    """Checks if the string ends with 'SM' (case-insensitive)."""
    return series.astype(str).str.strip().str.upper().str.endswith("SM")


def find_quantity_column(columns):
    """Dynamically finds the quantity column name regardless of casing/naming."""
    common_names = ["quantity", "qty", "order qty", "pcs", "total qty"]
    for col in columns:
        if col.strip().lower() in common_names:
            return col
    # Fallback search if exact match isn't found
    for col in columns:
        if "qty" in col.lower() or "quantity" in col.lower():
            return col
    return None


def process_jewelry_files(input_filepath, master_filepath):
    # 1. Dynamically locate header rows
    input_header_idx = find_header_row(
        input_filepath, "Elegant Jewelry Style #"
    )
    master_header_idx = find_header_row(master_filepath, "Style No")

    # 2. Load the Excel files using detected header rows
    input_df = pd.read_excel(input_filepath, header=input_header_idx)
    master_df = pd.read_excel(master_filepath, header=master_header_idx)

    # 3. Clean column headers (strip spaces)
    input_df.columns = input_df.columns.astype(str).str.strip()
    master_df.columns = master_df.columns.astype(str).str.strip()

    # 4. Find the Quantity column in input file
    qty_col = find_quantity_column(input_df.columns)
    if not qty_col:
        raise KeyError(
            "Could not identify a 'Quantity' or 'QTY' column in the input file."
        )

    # 5. Strictly normalize Input DataFrame keys
    input_df["match_style"] = clean_style(input_df["Elegant Jewelry Style #"])
    input_df["match_color"] = clean_color(input_df["Metal Color"])
    input_df["match_size"] = clean_size(input_df["Size"])

    # Extract SM suffix condition from 'OMJ Style #' if present
    if "OMJ Style #" in input_df.columns:
        input_df["match_sm"] = has_sm_suffix(input_df["OMJ Style #"])
    else:
        input_df["match_sm"] = False

    # 6. Strictly normalize Master DataFrame keys
    master_df["match_style"] = clean_style(master_df["Style No"])
    master_df["match_color"] = clean_color(master_df["COLOR"])
    master_df["match_size"] = clean_size(master_df["SIZE"])

    # Extract SM suffix condition from Master's 'Client Style No'
    master_df["match_sm"] = has_sm_suffix(master_df["Client Style No"])

    # 7. Remove duplicate keys from Master (including match_sm)
    merge_keys = ["match_style", "match_color", "match_size", "match_sm"]

    master_cleaned = master_df[
        merge_keys + ["Client Style No"]
    ].drop_duplicates(subset=merge_keys, keep="first")

    # 8. Merge on all 4 keys (Style, Color, Size, SM condition)
    merged_df = pd.merge(input_df, master_cleaned, on=merge_keys, how="left")

    # 9. Return both Quantity and Client Style No columns
    return merged_df[[qty_col, "Client Style No"]]


# Example Usage
if __name__ == "__main__":
    result_df = process_jewelry_files(
        input_filepath=r"D:\ops\tools\data\Om jewelery\PO# STOCK9925.xlsx",
        master_filepath=r"C:\Users\Pratik.SJFS\Downloads\OMJ_CS_1408.xlsx",
    )

    print(result_df)

    Quantity Client Style No
0       53.0     YR4171B-7WG
1       58.0     YR4171C-7WG
2       47.0     YR4171E-7WG
3       42.0     YR4171Z-7WG
4       41.0     YR4171F-7WG
..       ...             ...
59       NaN             NaN
60       NaN             NaN
61       NaN             NaN
62       NaN             NaN
63       NaN             NaN

[64 rows x 2 columns]


# working code for client style code and qunatity

In [ ]:
#working code
import pandas as pd


def find_header_row(filepath, key_column_name):
    """Reads the raw Excel sheet without headers and searches for the row that

    contains the specified key column name.
    """
    raw_df = pd.read_excel(filepath, header=None)

    for row_idx, row in raw_df.iterrows():
        row_values = row.astype(str).str.strip().values
        if key_column_name in row_values:
            return row_idx

    raise ValueError(
        f"Could not find header '{key_column_name}' in file: {filepath}"
    )


# --- Strict Cleaning Helpers ---
def clean_style(series):
    """Trims whitespace and converts style numbers to uppercase."""
    return series.astype(str).str.strip().str.upper()


def clean_size(series):
    """Trims whitespace and converts float strings like '7.0' to '7'."""
    return (
        series.astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.lower()
    )


def clean_color(series):
    """Standardizes color names and abbreviations.

    If two colors are detected (e.g., 'yellow white gold', 'white/yellow', 'two
    tone'), maps to 'tt'.
    """

    def parse_color_value(val):
        val = str(val).strip().lower()

        # 1. Direct two-tone / multi-color checks
        if any(
            kw in val
            for kw in ["two tone", "two-tone", "tt", "two color", "two-color"]
        ):
            return "tt"

        # 2. Check for presence of multiple distinct color keywords
        color_families = {
            "w": ["white", "wg"],
            "y": ["yellow", "yg"],
            "p": ["pink", "rose", "pg", "rg"],
            "pt": ["platinum", "plat"],
        }

        found_families = set()
        for family_code, keywords in color_families.items():
            for kw in keywords:
                if kw in val:
                    found_families.add(family_code)
                    break

        # If two or more distinct color families are found, classify as Two-Tone ('tt')
        if len(found_families) >= 2:
            return "tt"

        # 3. Standard single-color mapping dictionary
        color_map = {
            "white gold": "w",
            "white": "w",
            "wg": "w",
            "w": "w",
            "yellow gold": "y",
            "yellow": "y",
            "yg": "y",
            "y": "y",
            "pink gold": "p",
            "rose gold": "p",
            "pg": "p",
            "rg": "p",
            "p": "p",
            "platinum": "pt",
            "plat": "pt",
            "pt": "pt",
            "alloy": "al",
            "al": "al",
        }

        return color_map.get(val, val)

    return series.apply(parse_color_value)


def has_sm_suffix(series):
    """Checks if the string ends with 'SM' (case-insensitive)."""
    return series.astype(str).str.strip().str.upper().str.endswith("SM")


def find_quantity_column(columns):
    """Dynamically finds the quantity column name regardless of casing/naming."""
    common_names = ["quantity", "qty", "order qty", "pcs", "total qty"]
    for col in columns:
        if col.strip().lower() in common_names:
            return col

    # Fallback search if exact match isn't found
    for col in columns:
        if "qty" in col.lower() or "quantity" in col.lower():
            return col
    return None


def process_jewelry_files(input_filepath, master_filepath):
    # 1. Dynamically locate header rows
    input_header_idx = find_header_row(
        input_filepath, "Elegant Jewelry Style #"
    )
    master_header_idx = find_header_row(master_filepath, "Style No")

    # 2. Load the Excel files using detected header rows
    input_df = pd.read_excel(input_filepath, header=input_header_idx)
    master_df = pd.read_excel(master_filepath, header=master_header_idx)

    # 3. Clean column headers (strip spaces)
    input_df.columns = input_df.columns.astype(str).str.strip()
    master_df.columns = master_df.columns.astype(str).str.strip()

    # 4. Find the Quantity column in input file
    qty_col = find_quantity_column(input_df.columns)
    if not qty_col:
        raise KeyError(
            "Could not identify a 'Quantity' or 'QTY' column in the input file."
        )

    # 5. Strictly normalize Input DataFrame keys
    input_df["match_style"] = clean_style(input_df["Elegant Jewelry Style #"])
    input_df["match_color"] = clean_color(input_df["Metal Color"])
    input_df["match_size"] = clean_size(input_df["Size"])

    # Extract SM suffix condition from 'OMJ Style #' if present
    if "OMJ Style #" in input_df.columns:
        input_df["match_sm"] = has_sm_suffix(input_df["OMJ Style #"])
    else:
        input_df["match_sm"] = False

    # 6. Strictly normalize Master DataFrame keys
    master_df["match_style"] = clean_style(master_df["Style No"])
    master_df["match_color"] = clean_color(master_df["COLOR"])
    master_df["match_size"] = clean_size(master_df["SIZE"])

    # Extract SM suffix condition from Master's 'Client Style No'
    master_df["match_sm"] = has_sm_suffix(master_df["Client Style No"])

    # 7. Remove duplicate keys from Master (including match_sm)
    merge_keys = ["match_style", "match_color", "match_size", "match_sm"]

    master_cleaned = master_df[
        merge_keys + ["Client Style No"]
    ].drop_duplicates(subset=merge_keys, keep="first")

    # 8. Merge on all keys (Style, Color, Size, SM condition)
    merged_df = pd.merge(input_df, master_cleaned, on=merge_keys, how="left")

    # 9. Return Quantity and Client Style No columns
    return merged_df[[qty_col, "Client Style No"]]


# Example Usage
if __name__ == "__main__":
    result_df = process_jewelry_files(
        input_filepath=r"D:\latest\omj\Copy of PO# VAL81126.xlsx",
        master_filepath=r"C:\Users\Pratik.SJFS\Downloads\OMJ_CS_1408.xlsx",
    )

    print(result_df)

    Quantity Client Style No
0        NaN             NaN
1        NaN             NaN
2       25.0     YR4171E-7WG
3       20.0     YR4171F-7WG
4        NaN             NaN
5        NaN             NaN
6        NaN             NaN
7        NaN             NaN
8        NaN             NaN
9        NaN             NaN
10       NaN             NaN
11       NaN             NaN
12       NaN             NaN
13       NaN             NaN
14       NaN             NaN
15       NaN             NaN
16       NaN             NaN
17       NaN             NaN
18       NaN             NaN
19       NaN             NaN
20       NaN             NaN


In [21]:
import pandas as pd


def find_header_row(filepath, key_column_name):
    """Reads the raw Excel sheet without headers and searches for the row that

    contains the specified key column name.
    """
    raw_df = pd.read_excel(filepath, header=None)

    for row_idx, row in raw_df.iterrows():
        row_values = row.astype(str).str.strip().values
        if key_column_name in row_values:
            return row_idx

    raise ValueError(
        f"Could not find header '{key_column_name}' in file: {filepath}"
    )


# --- Strict Cleaning Helpers ---
def clean_style(series):
    """Trims whitespace and converts style numbers to uppercase."""
    return series.astype(str).str.strip().str.upper()


def clean_size(series):
    """Trims whitespace and converts float strings like '7.0' to '7'."""
    return (
        series.astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.lower()
    )


def clean_color(series):
    """Standardizes color names and abbreviations.

    If two colors are detected (e.g., 'yellow white gold', 'white/yellow', 'two
    tone'), maps to 'tt'.
    """

    def parse_color_value(val):
        val = str(val).strip().lower()

        # 1. Direct two-tone / multi-color checks
        if any(
            kw in val
            for kw in ["two tone", "two-tone", "tt", "two color", "two-color"]
        ):
            return "tt"

        # 2. Check for presence of multiple distinct color keywords
        color_families = {
            "w": ["white", "wg"],
            "y": ["yellow", "yg"],
            "p": ["pink", "rose", "pg", "rg"],
            "pt": ["platinum", "plat"],
        }

        found_families = set()
        for family_code, keywords in color_families.items():
            for kw in keywords:
                if kw in val:
                    found_families.add(family_code)
                    break

        # If two or more distinct color families are found, classify as Two-Tone ('tt')
        if len(found_families) >= 2:
            return "tt"

        # 3. Standard single-color mapping dictionary
        color_map = {
            "white gold": "w",
            "white": "w",
            "wg": "w",
            "w": "w",
            "yellow gold": "y",
            "yellow": "y",
            "yg": "y",
            "y": "y",
            "pink gold": "p",
            "rose gold": "p",
            "pg": "p",
            "rg": "p",
            "p": "p",
            "platinum": "pt",
            "plat": "pt",
            "pt": "pt",
            "alloy": "al",
            "al": "al",
        }

        return color_map.get(val, val)

    return series.apply(parse_color_value)


def has_sm_suffix(series):
    """Checks if the string ends with 'SM' (case-insensitive)."""
    return series.astype(str).str.strip().str.upper().str.endswith("SM")


def find_quantity_column(columns):
    """Dynamically finds the quantity column name regardless of casing/naming."""
    common_names = ["quantity", "qty", "order qty", "pcs", "total qty"]
    for col in columns:
        if col.strip().lower() in common_names:
            return col

    # Fallback search if exact match isn't found
    for col in columns:
        if "qty" in col.lower() or "quantity" in col.lower():
            return col
    return None


def process_jewelry_files(input_filepath, master_filepath):
    # 1. Dynamically locate header rows
    input_header_idx = find_header_row(
        input_filepath, "Elegant Jewelry Style #"
    )
    master_header_idx = find_header_row(master_filepath, "Style No")

    # 2. Load the Excel files using detected header rows
    input_df = pd.read_excel(input_filepath, header=input_header_idx)
    master_df = pd.read_excel(master_filepath, header=master_header_idx)

    # 3. Clean column headers (strip spaces)
    input_df.columns = input_df.columns.astype(str).str.strip()
    master_df.columns = master_df.columns.astype(str).str.strip()

    # 4. Find the Quantity column in input file
    qty_col = find_quantity_column(input_df.columns)
    if not qty_col:
        raise KeyError(
            "Could not identify a 'Quantity' or 'QTY' column in the input file."
        )

    # 5. Strictly normalize Input DataFrame keys
    input_df["match_style"] = clean_style(input_df["Elegant Jewelry Style #"])
    input_df["match_color"] = clean_color(input_df["Metal Color"])
    input_df["match_size"] = clean_size(input_df["Size"])

    # Extract SM suffix condition from 'OMJ Style #' if present
    if "OMJ Style #" in input_df.columns:
        input_df["match_sm"] = has_sm_suffix(input_df["OMJ Style #"])
    else:
        input_df["match_sm"] = False

    # 6. Strictly normalize Master DataFrame keys
    master_df["match_style"] = clean_style(master_df["Style No"])
    master_df["match_color"] = clean_color(master_df["COLOR"])
    master_df["match_size"] = clean_size(master_df["SIZE"])

    # Extract SM suffix condition from Master's 'Client Style No'
    master_df["match_sm"] = has_sm_suffix(master_df["Client Style No"])

    # 7. Remove duplicate keys from Master (including match_sm)
    merge_keys = ["match_style", "match_color", "match_size", "match_sm"]

    master_cleaned = master_df[
        merge_keys + ["Client Style No"]
    ].drop_duplicates(subset=merge_keys, keep="first")

    # 8. Merge on all keys (Style, Color, Size, SM condition)
    merged_df = pd.merge(input_df, master_cleaned, on=merge_keys, how="left")

    # 9. Extract output and filter out NaN / missing rows
    output_df = merged_df[[qty_col, "Client Style No"]].dropna(
        subset=["Client Style No"]
    )

    return output_df


# Example Usage
if __name__ == "__main__":
    result_df = process_jewelry_files(
        input_filepath=r"D:\ops\tools\data\Om jewelery\PO# STOCK9925.xlsx",
        master_filepath=r"C:\Users\Pratik.SJFS\Downloads\OMJ_CS_1408.xlsx",
    )

    print(result_df)

    Quantity   Client Style No
0       53.0       YR4171B-7WG
1       58.0       YR4171C-7WG
2       47.0       YR4171E-7WG
3       42.0       YR4171Z-7WG
4       41.0       YR4171F-7WG
5       38.0           YR4171X
6       27.0    YR4171XA-7.5WG
7       30.0       YR4171G-7WG
8       26.0       YR4171H-7WG
9       22.0      YR4171SA-7WG
10      17.0     YR4171SB-7WGM
11      20.0    RG0003614B-7WG
12      26.0    RG0003614C-7WG
13      23.0    RG0003614E-7WG
14      24.0    RG0003614Z-7WG
15      18.0    RG0003614F-7WG
16       2.0    RG0003614X-7WG
17      16.0   RG0003614XA-7WG
18      17.0    RG0003614G-7WG
19       0.0    RG0003614H-7WG
20      10.0    RG0000361C-7WG
21      10.0    RG0000361E-7WG
22      10.0    RG0000361Z-7WG
23       5.0     YR4321BMC-WGM
24       5.0     YR4321CMC-7WG
25       5.0     YR4321EMC-7WG
26       5.0     YR4321ZMC-7WG
27       5.0     YR4321FMC-WGM
28       3.0     YR4321XMC-7WG
29       3.0     YR4321GMC-7WG
30       2.0     YR4321SMC-7WG
31      

In [24]:
import pandas as pd


def find_header_row(filepath, key_column_name):
    """Reads the raw Excel sheet without headers and searches for the row that

    contains the specified key column name.
    """
    raw_df = pd.read_excel(filepath, header=None)

    for row_idx, row in raw_df.iterrows():
        row_values = row.astype(str).str.strip().values
        if key_column_name in row_values:
            return row_idx

    raise ValueError(
        f"Could not find header '{key_column_name}' in file: {filepath}"
    )


# --- Strict Cleaning Helpers ---
def clean_style(series):
    """Trims whitespace and converts style numbers to uppercase."""
    return series.astype(str).str.strip().str.upper()


def clean_size(series):
    """Trims whitespace and converts float strings like '7.0' to '7'."""
    return (
        series.astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.lower()
    )


def clean_color(series):
    """Standardizes color names and abbreviations.

    If two colors are detected (e.g., 'yellow white gold', 'white/yellow', 'two
    tone'), maps to 'tt'.
    """

    def parse_color_value(val):
        val = str(val).strip().lower()

        # 1. Direct two-tone / multi-color checks
        if any(
            kw in val
            for kw in ["two tone", "two-tone", "tt", "two color", "two-color"]
        ):
            return "tt"

        # 2. Check for presence of multiple distinct color keywords
        color_families = {
            "w": ["white", "wg"],
            "y": ["yellow", "yg"],
            "p": ["pink", "rose", "pg", "rg"],
            "pt": ["platinum", "plat"],
        }

        found_families = set()
        for family_code, keywords in color_families.items():
            for kw in keywords:
                if kw in val:
                    found_families.add(family_code)
                    break

        # If two or more distinct color families are found, classify as Two-Tone ('tt')
        if len(found_families) >= 2:
            return "tt"

        # 3. Standard single-color mapping dictionary
        color_map = {
            "white gold": "w",
            "white": "w",
            "wg": "w",
            "w": "w",
            "yellow gold": "y",
            "yellow": "y",
            "yg": "y",
            "y": "y",
            "pink gold": "p",
            "rose gold": "p",
            "pg": "p",
            "rg": "p",
            "p": "p",
            "platinum": "pt",
            "plat": "pt",
            "pt": "pt",
            "alloy": "al",
            "al": "al",
        }

        return color_map.get(val, val)

    return series.apply(parse_color_value)


def has_sm_suffix(series):
    """Checks if the string ends with 'SM' (case-insensitive)."""
    return series.astype(str).str.strip().str.upper().str.endswith("SM")


def find_quantity_column(columns):
    """Dynamically finds the quantity column name regardless of casing/naming."""
    common_names = ["quantity", "qty", "order qty", "pcs", "total qty"]
    for col in columns:
        if col.strip().lower() in common_names:
            return col

    # Fallback search if exact match isn't found
    for col in columns:
        if "qty" in col.lower() or "quantity" in col.lower():
            return col
    return None


def process_jewelry_files(input_filepath, master_filepath):
    # 1. Dynamically locate header rows
    input_header_idx = find_header_row(
        input_filepath, "Elegant Jewelry Style #"
    )
    master_header_idx = find_header_row(master_filepath, "Style No")

    # 2. Load the Excel files using detected header rows
    input_df = pd.read_excel(input_filepath, header=input_header_idx)
    master_df = pd.read_excel(master_filepath, header=master_header_idx)

    # 3. Clean column headers (strip spaces)
    input_df.columns = input_df.columns.astype(str).str.strip()
    master_df.columns = master_df.columns.astype(str).str.strip()

    # 4. Find the Quantity column in input file
    qty_col = find_quantity_column(input_df.columns)
    if not qty_col:
        raise KeyError(
            "Could not identify a 'Quantity' or 'QTY' column in the input file."
        )

    # Assign SKUNo from 'OMJ Style #'
    if "OMJ Style #" in input_df.columns:
        input_df["SKUNo"] = input_df["OMJ Style #"].astype(str).str.strip()
    else:
        input_df["SKUNo"] = None

    # Set default OrderItemPcs to '1' for all input rows
    input_df["OrderItemPcs"] = "1"

    # 5. Strictly normalize Input DataFrame keys
    input_df["match_style"] = clean_style(input_df["Elegant Jewelry Style #"])
    input_df["match_color"] = clean_color(input_df["Metal Color"])
    input_df["match_size"] = clean_size(input_df["Size"])

    # Extract SM suffix condition from 'OMJ Style #' if present
    if "OMJ Style #" in input_df.columns:
        input_df["match_sm"] = has_sm_suffix(input_df["OMJ Style #"])
    else:
        input_df["match_sm"] = False

    # 6. Strictly normalize Master DataFrame keys
    master_df["match_style"] = clean_style(master_df["Style No"])
    master_df["match_color"] = clean_color(master_df["COLOR"])
    master_df["match_size"] = clean_size(master_df["SIZE"])

    # Extract SM suffix condition from Master's 'Client Style No'
    master_df["match_sm"] = has_sm_suffix(master_df["Client Style No"])

    # 7. List of required Master columns to bring over
    master_requested_cols = [
        "Client Style No",
        "COLOR",
        "ItemSize",
        "Base Metal",
        "SpecialRemarks",
        "CustomerProductionInstruction",
        "DesignProductionInstruction",
        "Stamping Instruction",
    ]

    # Keep only those master requested columns that actually exist in master_df
    available_master_cols = [
        col for col in master_requested_cols if col in master_df.columns
    ]

    # Remove duplicate keys from Master (including match_sm)
    merge_keys = ["match_style", "match_color", "match_size", "match_sm"]

    master_cleaned = master_df[
        merge_keys + available_master_cols
    ].drop_duplicates(subset=merge_keys, keep="first")

    # 8. Merge on all keys (Style, Color, Size, SM condition)
    merged_df = pd.merge(input_df, master_cleaned, on=merge_keys, how="left")

    # 9. Define final column order and drop missing/unmatched rows
    final_columns = ["SKUNo", qty_col, "OrderItemPcs"] + available_master_cols

    output_df = merged_df[final_columns].dropna(subset=["Client Style No"])

    return output_df


# Example Usage
if __name__ == "__main__":
    result_df = process_jewelry_files(
        input_filepath=r"D:\latest\omj\Copy of PO# VAL81126.xlsx",
        master_filepath=r"C:\Users\Pratik.SJFS\Downloads\OMJ_CS_1408.xlsx",
    )

    print(result_df)

        SKUNo  Quantity OrderItemPcs Client Style No COLOR ItemSize  \
2  VAL1519-30      25.0            1     YR4171E-7WG     W     UP07   
3  VAL1915-50      20.0            1     YR4171F-7WG     W     UP07   

  Base Metal                                     SpecialRemarks  \
2       G14W  VAL1519-30,14K WHITE GOLD,RING SIZE 7,WHITE RH...   
3       G14W  VAL1915-50,14K WHITE GOLD,Size:7,DIA QLTY:I W5...   

                       CustomerProductionInstruction  \
2  *PLEASE MAKE SURE NO POROSITY,  PLEASE MAKE SU...   
3  PLEASE MAKE SURE NO POROSITY, PLEASE MAKE SURE...   

                         DesignProductionInstruction  \
2  WHITE RHODIUM,PERFECT MM SIZE OF THE RING AS M...   
3  WHITE RHODIUM, STRICT QC QUALITY, PERFECT MM S...   

            Stamping Instruction  
2   OMJ LOGO,14K,7,VAL1519-30,E.  
3  OMJ LOGO,14K,7,VAL1915-50, E.  


In [26]:
import pandas as pd


def find_header_row(filepath, key_column_name):
    """Reads the raw Excel sheet without headers and searches for the row that

    contains the specified key column name.
    """
    raw_df = pd.read_excel(filepath, header=None)

    for row_idx, row in raw_df.iterrows():
        row_values = row.astype(str).str.strip().values
        if key_column_name in row_values:
            return row_idx

    raise ValueError(
        f"Could not find header '{key_column_name}' in file: {filepath}"
    )


# --- Strict Cleaning Helpers ---
def clean_style(series):
    """Trims whitespace and converts style numbers to uppercase."""
    return series.astype(str).str.strip().str.upper()


def clean_size(series):
    """Trims whitespace and converts float strings like '7.0' to '7'."""
    return (
        series.astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.lower()
    )


def clean_color(series):
    """Standardizes color names and abbreviations.

    If two colors are detected (e.g., 'yellow white gold', 'white/yellow', 'two
    tone'), maps to 'tt'.
    """

    def parse_color_value(val):
        val = str(val).strip().lower()

        # 1. Direct two-tone / multi-color checks
        if any(
            kw in val
            for kw in ["two tone", "two-tone", "tt", "two color", "two-color"]
        ):
            return "tt"

        # 2. Check for presence of multiple distinct color keywords
        color_families = {
            "w": ["white", "wg"],
            "y": ["yellow", "yg"],
            "p": ["pink", "rose", "pg", "rg"],
            "pt": ["platinum", "plat"],
        }

        found_families = set()
        for family_code, keywords in color_families.items():
            for kw in keywords:
                if kw in val:
                    found_families.add(family_code)
                    break

        if len(found_families) >= 2:
            return "tt"

        # 3. Standard single-color mapping dictionary
        color_map = {
            "white gold": "w",
            "white": "w",
            "wg": "w",
            "w": "w",
            "yellow gold": "y",
            "yellow": "y",
            "yg": "y",
            "y": "y",
            "pink gold": "p",
            "rose gold": "p",
            "pg": "p",
            "rg": "p",
            "p": "p",
            "platinum": "pt",
            "plat": "pt",
            "pt": "pt",
            "alloy": "al",
            "al": "al",
        }

        return color_map.get(val, val)

    return series.apply(parse_color_value)


def has_sm_suffix(series):
    """Checks if the string ends with 'SM' (case-insensitive)."""
    return series.astype(str).str.strip().str.upper().str.endswith("SM")


def find_quantity_column(columns):
    """Dynamically finds the quantity column name regardless of casing/naming."""
    common_names = ["quantity", "qty", "order qty", "pcs", "total qty"]
    for col in columns:
        if col.strip().lower() in common_names:
            return col

    for col in columns:
        if "qty" in col.lower() or "quantity" in col.lower():
            return col
    return None


def process_jewelry_files(input_filepath, master_filepath, item_po_no=""):
    # 1. Dynamically locate header rows
    input_header_idx = find_header_row(
        input_filepath, "Elegant Jewelry Style #"
    )
    master_header_idx = find_header_row(master_filepath, "Style No")

    # 2. Load the Excel files using detected header rows
    input_df = pd.read_excel(input_filepath, header=input_header_idx)
    master_df = pd.read_excel(master_filepath, header=master_header_idx)

    # 3. Clean column headers (strip spaces)
    input_df.columns = input_df.columns.astype(str).str.strip()
    master_df.columns = master_df.columns.astype(str).str.strip()

    # 4. Find the Quantity column in input file
    qty_col = find_quantity_column(input_df.columns)
    if not qty_col:
        raise KeyError(
            "Could not identify a 'Quantity' or 'QTY' column in the input file."
        )

    # Assign SKUNo from 'OMJ Style #'
    if "OMJ Style #" in input_df.columns:
        input_df["SKUNo"] = input_df["OMJ Style #"].astype(str).str.strip()
    else:
        input_df["SKUNo"] = ""

    # Set default OrderItemPcs to '1'
    input_df["OrderItemPcs"] = "1"

    # 5. Strictly normalize Input DataFrame keys
    input_df["match_style"] = clean_style(input_df["Elegant Jewelry Style #"])
    input_df["match_color"] = clean_color(input_df["Metal Color"])
    input_df["match_size"] = clean_size(input_df["Size"])

    if "OMJ Style #" in input_df.columns:
        input_df["match_sm"] = has_sm_suffix(input_df["OMJ Style #"])
    else:
        input_df["match_sm"] = False

    # 6. Strictly normalize Master DataFrame keys
    master_df["match_style"] = clean_style(master_df["Style No"])
    master_df["match_color"] = clean_color(master_df["COLOR"])
    master_df["match_size"] = clean_size(master_df["SIZE"])
    master_df["match_sm"] = has_sm_suffix(master_df["Client Style No"])

    # 7. List of required Master columns to bring over
    master_requested_cols = [
        "Client Style No",
        "COLOR",
        "ItemSize",
        "Base Metal",
        "SpecialRemarks",
        "CustomerProductionInstruction",
        "DesignProductionInstruction",
        "Stamping Instruction",
    ]

    available_master_cols = [
        col for col in master_requested_cols if col in master_df.columns
    ]
    merge_keys = ["match_style", "match_color", "match_size", "match_sm"]

    master_cleaned = master_df[
        merge_keys + available_master_cols
    ].drop_duplicates(subset=merge_keys, keep="first")

    # 8. Merge on all keys (Style, Color, Size, SM condition)
    merged_df = pd.merge(input_df, master_cleaned, on=merge_keys, how="left")

    # Drop missing matches
    df = merged_df.dropna(subset=["Client Style No"]).copy()

    # 9. Map/Rename existing columns
    df["StyleCode"] = df["Client Style No"]
    df["OrderQty"] = df[qty_col]
    df["Metal"] = df["Base Metal"] if "Base Metal" in df.columns else ""
    df["Tone"] = df["COLOR"] if "COLOR" in df.columns else ""
    df["StampInstruction"] = (
        df["Stamping Instruction"]
        if "Stamping Instruction" in df.columns
        else ""
    )
    df["ItemPoNo"] = item_po_no

    # Ensure optional master columns exist if missing from sheet
    for col in [
        "ItemSize",
        "CustomerProductionInstruction",
        "SpecialRemarks",
        "DesignProductionInstruction",
    ]:
        if col not in df.columns:
            df[col] = ""

    # 10. Generate Auto-incremental SrNo
    df["SrNo"] = range(1, len(df) + 1)

    # 11. Add Blank Columns
    blank_columns = [
        "ItemRefNo",
        "StockType",
        "MakeType",
        "OrderGroup",
        "Certificate",
        "Basestoneminwt",
        "Basestonemaxwt",
        "Basemetalminwt",
        "Basemetalmaxwt",
        "Productiondeliverydate",
        "Expecteddeliverydate",
        "BlankColumn",
        "SetPrice",
        "StoneQuality",
        "Date",
        "PoDate",
        "E Del Date",
    ]
    for col in blank_columns:
        df[col] = ""

    # 12. Final Ordered Schema
    final_ordered_columns = [
        "SrNo",
        "StyleCode",
        "ItemSize",
        "OrderQty",
        "OrderItemPcs",
        "Metal",
        "Tone",
        "ItemPoNo",
        "ItemRefNo",
        "StockType",
        "MakeType",
        "CustomerProductionInstruction",
        "SpecialRemarks",
        "DesignProductionInstruction",
        "StampInstruction",
        "OrderGroup",
        "Certificate",
        "SKUNo",
        "Basestoneminwt",
        "Basestonemaxwt",
        "Basemetalminwt",
        "Basemetalmaxwt",
        "Productiondeliverydate",
        "Expecteddeliverydate",
        "BlankColumn",
        "SetPrice",
        "StoneQuality",
        "Date",
        "PoDate",
        "E Del Date",
    ]

    return df[final_ordered_columns]


# Example Usage
if __name__ == "__main__":
    po_input = input("Enter Item PO Number: ") or "PO-998877"

    result_df = process_jewelry_files(
        input_filepath=r"D:\latest\omj\Copy of PO# VAL81126.xlsx",
        master_filepath=r"C:\Users\Pratik.SJFS\Downloads\OMJ_CS_1408.xlsx",
        item_po_no=po_input,
    )

    print(result_df)
    result_df.to_excel('output3.xlsx')

   SrNo    StyleCode ItemSize  OrderQty OrderItemPcs Metal Tone ItemPoNo  \
2     1  YR4171E-7WG     UP07      25.0            1  G14W    W   109090   
3     2  YR4171F-7WG     UP07      20.0            1  G14W    W   109090   

  ItemRefNo StockType  ... Basemetalminwt Basemetalmaxwt  \
2                      ...                                 
3                      ...                                 

  Productiondeliverydate Expecteddeliverydate BlankColumn SetPrice  \
2                                                                    
3                                                                    

  StoneQuality Date PoDate E Del Date  
2                                      
3                                      

[2 rows x 30 columns]
